# EDA — AIOps Log Monitor

Análisis exploratorio del dataset de logs (`data/sample.csv`).

Cubre: valores faltantes, rangos numéricos, distribución de la variable objetivo (status/severity), conteo de categóricas, matriz de correlación, palabras frecuentes en `message` y feature importance para predecir fallos.

Este notebook se exporta a `scripts/eda.py` para ejecución reproducible.

In [ ]:
import os
import re
from collections import Counter

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

DATA_PATH = os.path.join('..', 'data', 'processed_events_analysis.csv')
FALLBACK = os.path.join('..', 'data', 'sample.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = FALLBACK
OUT_DIR = os.path.join('..', 'docs', 'eda')
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True

def save_fig(name):
    path = os.path.join(OUT_DIR, name)
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print('saved', path)

## 1. Carga y valores faltantes

In [ ]:
df = pd.read_csv(DATA_PATH)
print('shape', df.shape)
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing.to_string())

fig, ax = plt.subplots()
if not missing.empty:
    missing.plot(kind='barh', ax=ax, color='#4c72b0')
    ax.set_xlabel('Count of missing values'); ax.set_title('Missing values per column')
else:
    ax.text(0.5, 0.5, 'No missing values', ha='center', va='center')
save_fig('missing_values.png'); plt.show()

## 2. Rangos de variables numéricas (min/max/mean/std)

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
desc = df[numeric_cols].describe().loc[['min', 'max', 'mean', 'std']]
print(desc.to_string())

key_numeric = [c for c in ['duration_ms','status_code','hour','event_count','line','process_id'] if c in df.columns]
fig, axes = plt.subplots(2, 3, figsize=(15, 8)); axes = axes.ravel()
for i, col in enumerate(key_numeric[:6]):
    df[col].hist(ax=axes[i], bins=30, color='#55a868', edgecolor='black')
    axes[i].set_title(col)
for j in range(len(key_numeric), len(axes)):
    axes[j].axis('off')
save_fig('numeric_distributions.png'); plt.show()

## 3. Variable objetivo: status y severity

In [ ]:
for col in ['status', 'severity', 'event_type', 'category', 'environment']:
    if col in df.columns:
        print('===', col, '===')
        print(df[col].value_counts(dropna=False).to_string())
        fig, ax = plt.subplots()
        df[col].value_counts().head(10).plot(kind='bar', ax=ax, color='#8172b3')
        ax.set_title(col + ' distribution'); plt.xticks(rotation=45, ha='right')
        save_fig(col + '_distribution.png'); plt.show()

failure_rate = (df['status'].astype(str).str.lower() == 'failure').mean()
print('Failure rate:', round(failure_rate, 4))

## 4. Matriz de correlación

In [ ]:
corr = df[numeric_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns))); ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticklabels(corr.index, fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Correlation matrix'); save_fig('correlation_matrix.png'); plt.show()

## 5. Palabras frecuentes en los mensajes de log

In [ ]:
messages = df['message'].dropna().astype(str)
words = Counter()
for msg in messages:
    words.update(re.findall(r'[a-zA-ZáéíóúñÁÉÍÓÚÑ0-9]+', msg.lower()))
top_words = words.most_common(20)
print(top_words)
if top_words:
    fig, ax = plt.subplots(figsize=(10, 6))
    ws, cs = zip(*top_words)
    ax.barh(range(len(ws)), cs, color='#c44e52')
    ax.set_yticks(range(len(ws))); ax.set_yticklabels(ws); ax.invert_yaxis()
    ax.set_title('Top frequent words in log messages')
    save_fig('message_word_frequencies.png'); plt.show()

## 6. Feature importance (RandomForest, objetivo = FAILURE)

In [ ]:
feature_cols = [c for c in ['duration_ms','status_code','hour','event_count','line','process_id','severity','event_type','category','environment','method'] if c in df.columns]
X = df[feature_cols].copy()
y = (df['status'].astype(str).str.lower() == 'failure').astype(int)
for col in X.columns:
    if X[col].dtype == object:
        X[col] = X[col].astype('category').cat.codes
X = X.fillna(0)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.to_string())
fig, ax = plt.subplots(figsize=(10, 6))
importance.head(10).plot(kind='barh', ax=ax, color='#64a0c8'); ax.invert_yaxis()
ax.set_title('Top 10 feature importance (predicting failure)')
save_fig('feature_importance.png'); plt.show()

## Conclusión

El dataset está mayormente completo en columnas clave (status, severity, event_type). Los nulos se concentran en columnas de request (message, method, path, status_code, duration_ms) que solo aplican a eventos HTTP. La tasa de fallos es ~27%. `event_type` y `severity` son las features más predictivas para el fallo. Todos los gráficos se guardan en `docs/eda/`.